# Lektion 09 - Metakognitions-Entwurfsmuster


## Einrichtung

Dieses Notebook demonstriert das Metacognition-Designmuster mit dem Microsoft Agent Framework.

**Voraussetzungen:**
- Azure OpenAI-Bereitstellung über Umgebungsvariablen konfiguriert
- Azure CLI authentifiziert (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Was ist Metakognition?

Metakognition ist **Denken über das Denken**. Im Kontext von KI-Agenten bedeutet es, Agenten zu entwickeln, die Folgendes können:

- **Sich selbst reflektieren** über ihre eigenen Ausgaben und ihren Denkprozess
- **Fehler erkennen** und sich elegant erholen, statt stillschweigend zu scheitern
- **Bewerten**, ob ihre Antworten vollständig und hilfreich sind
- **Ihre Strategie anpassen**, wenn ein erster Ansatz nicht funktioniert (z. B. auf ein Backup-System zurückgreifen)

Ein metakognitiver Agent beantwortet nicht nur Fragen — er überwacht seine eigene Leistung und passt sich in Echtzeit an.


## Primäre und Backup-Tools

Ein verbreitetes Metakognitionsmuster ist die **Fallback-Strategie**. Der Agent versucht zunächst ein primäres Tool; wenn dieses fehlschlägt (z. B. ein 404-Fehler), erkennt der Agent das Scheitern und wechselt transparent zu einem Backup-Tool.

Das spiegelt reale Systeme wider, in denen primäre Dienste möglicherweise nicht verfügbar sind und Agenten das Problem selbst diagnostizieren müssen, bevor sie einen alternativen Weg wählen.

Im Folgenden definieren wir zwei Flugabfrage-Tools:
- **Primary** — umfasst Paris, Tokio und Barcelona
- **Backup** — umfasst Berlin, Sydney und New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Selbstreflektierender Agent mit Fehlerwiederherstellung

Der untenstehende Agent wird angewiesen, zuerst das primäre Flugsystem zu verwenden, Ausfälle zu erkennen und transparent auf das Backup-System zurückzugreifen. Nach jeder Antwort bewertet er kurz selbst, ob er die Frage des Nutzers vollständig beantwortet hat.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Muster der Selbstbewertung

Ein weiterer Aspekt der Metakognition ist die **Selbstbewertung**: ein separater Agent (oder derselbe Agent bei einem zweiten Durchgang) überprüft eine Antwort auf Vollständigkeit, Genauigkeit und Nützlichkeit.

Im Folgenden erstellen wir einen `ResponseEvaluator`-Agenten, der Antworten von Reiseagenten in drei Dimensionen bewertet.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Zusammenfassung

In dieser Lektion haben Sie gelernt, wie man **metakognitive Agenten** mit dem Microsoft Agent Framework erstellt:

- **Selbstreflexion**: Agenten, die ihr eigenes Denken überwachen und transparent kommunizieren, was passiert ist.
- **Fehlerbehebung mit Fallbacks**: Ein Primär- + Backup-Tool-Pattern, bei dem der Agent Fehler (z. B. 404-Fehler) erkennt und automatisch eine alternative Quelle ausprobiert.
- **Selbstevaluation**: Ein separater Bewertungsagent, der Antworten hinsichtlich Vollständigkeit, Genauigkeit und Nützlichkeit bewertet.

Diese Muster machen Agenten robuster, transparenter und vertrauenswürdiger — entscheidende Eigenschaften für den Produktionseinsatz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, weisen wir darauf hin, dass automatische Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner ursprünglichen Sprache ist als maßgebliche Quelle zu betrachten. Für wichtige Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
